# Ames Housing - Step 1: Explore and Understand The Data

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

current = Path.cwd()
project_candidates = [current, current / "IE500618Assignment1", *current.parents]
project_root = next(
    candidate
    for candidate in project_candidates
    if (candidate / "src" / "config.py").exists()
)
sys.path.insert(0, str(project_root / "src"))

from config import PATH_AMESHOUSING

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv(PATH_AMESHOUSING)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Dataset path: {PATH_AMESHOUSING}")
display(df.head())

In [ ]:
# Overview of variables and data types
structure = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_missing": df.notna().sum(),
    "unique_values": df.nunique(dropna=False),
})

print(f"Numeric variables: {df.select_dtypes(include="number").shape[1]}")
print(f"Categorical variables: {df.select_dtypes(exclude="number").shape[1]}")
display(structure)

df.info()

## 2. Investigate Missing Values

In [ ]:
df_missing = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
      .assign(missing_percent=lambda x: 100 * x["missing_count"] / len(df))
      .query("missing_count > 0")
      .sort_values("missing_count", ascending=False)
)

display(df_missing.round(2))
print(f"Columns with at least one missing value: {len(df_missing)}")

In [ ]:
# Visualise the columns
top_missing = df_missing.head(10).sort_values("missing_percent")

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=top_missing,
    x="missing_percent",
    y=top_missing.index,
    color="#4C78A8",
)
ax.set_title("Columns with The Most Missing Values")
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("Column")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
plt.xlim(0, 105)
plt.tight_layout()
plt.show()

## 3. Explore `SalePrice`

In [ ]:
sale_price = df["SalePrice"]
display(sale_price.describe().to_frame().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(sale_price, bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of sale prices")
axes[0].set_xlabel("Sale Price (USD)")

sns.boxplot(x=sale_price, ax=axes[1])
axes[1].set_title("Sale-price boxplot")
axes[1].set_xlabel("Sale Price (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Most expensive houses
display(
    df[["SalePrice", "Gr Liv Area", "Overall Qual", "Year Built", "Neighborhood"]]
    .sort_values("SalePrice", ascending=False)
    .head(10)
)

## 4. Looking for Numerical Relationships

In [ ]:
numeric_df = df.select_dtypes(include="number").drop(
    columns=["Order", "PID"], errors="ignore")
sale_price_correlations = (
    numeric_df.corr(numeric_only=True)["SalePrice"]
    .drop("SalePrice")
    .sort_values(key=abs, ascending=False)
)

display(sale_price_correlations.head(15).round(
    3).to_frame("SalePrice_corr"))

In [ ]:
# Heatmap of SalePrice corr
top_numeric_features = sale_price_correlations.abs().head(
    10).index.tolist() + ["SalePrice"]
plt.figure(figsize=(9, 7))
sns.heatmap(
    df[top_numeric_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="Spectral",
    center=0,
)
plt.title("Correlations among SalePrice")
plt.tight_layout()
plt.show()

## 5. Plots of Important Relationships

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=df, x="Gr Liv Area", y="SalePrice",
                alpha=0.45, ax=axes[0, 0])
axes[0, 0].set_title("Living Area and Sale Price")

sns.boxplot(data=df, x="Overall Qual", y="SalePrice", ax=axes[0, 1])
axes[0, 1].set_title("Overall Quality and Sale Price")

sns.scatterplot(data=df, x="Year Built", y="SalePrice",
                alpha=0.45, ax=axes[1, 0])
axes[1, 0].set_title("Year Built and Sale Price")

sns.scatterplot(data=df, x="Garage Area", y="SalePrice",
                alpha=0.45, ax=axes[1, 1])
axes[1, 1].set_title("Garage Area and Sale Price")

for ax in axes.flat:
    ax.set_ylabel("Sale Price (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# Comparision of neighbourhoods
df_neighbourhood = (
    df.groupby("Neighborhood")["SalePrice"]
      .median()
      .sort_values()
      .index
)

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=df_neighbourhood,
)
plt.xticks(rotation=60, ha="right")
plt.title("Sale to Price Distributions by Neighbourhood")
plt.xlabel("Neighbourhood")
plt.ylabel("Sale Price (USD)")
plt.tight_layout()
plt.show()